# Shaped-Reward Poker REINFORCE — FAST Colab version

Trimmed version for the STAT 4830 final presentation. Signal check in ~15-25 min on Colab T4.

**What changed vs the full version:** baseline eval skipped (uses Exp B's leaderboard number directly), 10 RL iters instead of 20, eval does max_steps=1 with max_new_tokens=384 (vs max_steps=3, max_new_tokens=1024). Enough signal to tell if the shaped reward breaks the 20% flat line; not enough for a full results table. Run a longer version later if the signal is there.

**Shaped reward:**
```
base_reward  = action_type_match (0.0 or 1.0)
tool_bonus   = 0.3 if model wrote real code else 0.0
             + 0.2 if real code parsed opponent stats else 0.0
penalty      = 0.2 if model used the canned-action fallback else 0.0
reward       = base_reward + tool_bonus - penalty
```

### How to use

1. **Runtime → Change runtime type → T4 GPU** (required).
2. **Runtime → Run all**.
3. Scroll to the bottom leaderboard when done.

## 1. Setup

In [ ]:
!pip install -q transformers peft trl datasets accelerate 'bitsandbytes>=0.43'

In [ ]:
import os, sys

if not os.path.exists('STAT-4830-RL-project'):
    !git clone --depth 1 https://github.com/aryanarora236/STAT-4830-RL-project.git
os.chdir('STAT-4830-RL-project')
!git pull --ff-only

sys.path.insert(0, os.getcwd())
print('repo at', os.getcwd())

In [ ]:
import torch
assert torch.cuda.is_available(), 'No GPU. Runtime > Change runtime type > T4 GPU, then Runtime > Restart.'
gpu = torch.cuda.get_device_properties(0)
cap = torch.cuda.get_device_capability(0)
print(f"GPU: {gpu.name}  |  compute {cap[0]}.{cap[1]}  |  {gpu.total_memory / 1e9:.1f} GB VRAM")
print(f"Precision: {'bf16 (Ampere+)' if cap[0] >= 8 else 'fp16 (T4/Volta)'}")

## 2. Apply the shaped-reward patch

In [ ]:
path = 'src/poker/training.py'
with open(path, 'r') as f:
    src = f.read()

if 'SHAPED REWARD' in src:
    print('Patch already applied, skipping.')
else:
    patches = [
        (
            "        for _ in range(self.batch_size):\n            attempted += 1\n            context, question, correct_answer = self.task_generator()",
            "        for _ in range(self.batch_size):\n            attempted += 1\n            was_wrapped = False  # SHAPED REWARD\n            context, question, correct_answer = self.task_generator()",
        ),
        (
            "                    if attempt_idx == 2:\n                        wrapped_action_code_count += 1\n                        raw_type, raw_amt = parse_action(response_text)",
            "                    if attempt_idx == 2:\n                        wrapped_action_code_count += 1\n                        was_wrapped = True  # SHAPED REWARD\n                        raw_type, raw_amt = parse_action(response_text)",
        ),
        (
            "            reward = compute_poker_reward_simple(predicted, correct_answer)\n            if reward > 0:\n                nonzero_reward_count += 1",
            "            base_reward = compute_poker_reward_simple(predicted, correct_answer)\n"
            "            # SHAPED REWARD: reward real code + stat parsing, penalize fallback\n"
            "            tool_bonus = 0.0\n"
            "            if not was_wrapped:\n"
            "                tool_bonus += 0.3\n"
            "            if parsed_stats:\n"
            "                tool_bonus += 0.2\n"
            "            fallback_penalty = 0.2 if was_wrapped else 0.0\n"
            "            reward = base_reward + tool_bonus - fallback_penalty\n"
            "            if reward > 0:\n"
            "                nonzero_reward_count += 1",
        ),
    ]
    for i, (old, new) in enumerate(patches, 1):
        assert old in src, f'Patch {i} insertion point not found.'
        src = src.replace(old, new, 1)
    with open(path, 'w') as f:
        f.write(src)
    print('All 3 patches applied.')

## 3. Copy starting checkpoint + skip baseline eval

Exp B's `best_by_eval` leaderboard entry is already 20.0% on held-out — we use that number directly instead of re-running baseline eval (which would take 20-30 min silent on T4).

In [ ]:
import shutil

START_CKPT_SRC = 'docs/results/poker_rl_expB_mixed20_20260422/poker_rl_expB_mixed20_20260422/best_by_eval'
START_CKPT = 'checkpoints/poker_rl_expB_best'
os.makedirs('checkpoints', exist_ok=True)
if os.path.exists(START_CKPT):
    shutil.rmtree(START_CKPT)
shutil.copytree(START_CKPT_SRC, START_CKPT)

# Skip the ~20-30 min silent baseline eval — use Exp B's leaderboard number directly.
baseline_acc = 0.20
baseline_reward = 0.20
print(f'Starting checkpoint: {START_CKPT}')
print(f'Baseline (from Exp B leaderboard): {baseline_acc:.1%}')

## 4. Train 10 iters with shaped reward (~8-12 min on T4)

Saves every 5 iters → checkpoints at `iter_5` and `iter_10`.

**Watch in streaming output:** `real_code=X/4`. In Exp B this averaged 0.3/4. If shaping is working, you'll see 2/4, 3/4, 4/4.

In [ ]:
!PYTHONUNBUFFERED=1 python -u scripts/poker_train.py \
    --phase rl \
    --model ./checkpoints/poker_rl_expB_best \
    --seed 20260422 \
    --rl-iterations 10 \
    --rl-batch-size 4 \
    --rl-lr 5e-6 \
    --rl-sample-temperature 0.2 \
    --rl-top-p 0.9 \
    --rl-adv-clip 2.0 \
    --ema-gamma 0.9 \
    --max-new-tokens 512 \
    --rl-output ./checkpoints/poker_rl_shaped 2>&1 | tee shaped_run.log

## 5. Fast held-out eval on saved checkpoints

8 episodes per checkpoint, single-step agent, 384 max tokens. Fast but noisy — movement of 10+ percentage points is a real signal; movement of less is within noise at n=8.

In [ ]:
import random, glob, json
from pathlib import Path
from src.training import load_model
from src.poker.agents import PokerLocalLLMAgent
from src.poker.tasks import generate_poker_task
from src.poker.rewards import parse_action

EVAL_EPISODES = 8
EVAL_SEED = 20260422

def fast_eval(model, tokenizer):
    random.seed(EVAL_SEED)
    torch.manual_seed(EVAL_SEED)
    # FAST CONFIG: single REPL step, 384 max new tokens per generation
    agent = PokerLocalLLMAgent(
        model=model, tokenizer=tokenizer, name='eval',
        max_steps=1, max_new_tokens=384, temperature=0.1,
    )
    correct = 0
    for i in range(EVAL_EPISODES):
        context, question, answer = generate_poker_task()
        pred, _ = agent.run_episode(context, question, answer)
        pred_type = parse_action(pred)[0]
        corr_type = parse_action(answer)[0]
        match = 1 if pred_type == corr_type else 0
        correct += match
        print(f'  ep {i+1}/{EVAL_EPISODES}: predicted={pred!r}, correct={answer!r}, match={bool(match)}')
    return correct / EVAL_EPISODES

results = [{'iteration': 0, 'held_out_acc': baseline_acc, 'source': 'baseline (Exp B leaderboard)'}]

ckpt_dirs = sorted(
    glob.glob('checkpoints/poker_rl_shaped/iter_*'),
    key=lambda p: int(p.rsplit('_', 1)[-1])
)
print(f'Found {len(ckpt_dirs)} checkpoints: {[Path(d).name for d in ckpt_dirs]}')

for ckpt in ckpt_dirs:
    it = int(Path(ckpt).name.split('_')[-1])
    print(f'\n--- Evaluating {Path(ckpt).name} ({EVAL_EPISODES} eps) ---')
    model, tokenizer = load_model(ckpt, load_in_4bit=True)
    model.eval()
    acc = fast_eval(model, tokenizer)
    print(f'  iter {it}: held-out acc = {acc:.1%}')
    results.append({'iteration': it, 'held_out_acc': acc, 'source': ckpt})
    del model, tokenizer
    torch.cuda.empty_cache()

with open('shaped_eval_leaderboard.json', 'w') as f:
    json.dump(results, f, indent=2)

print('\n' + '='*60)
print('SHAPED-REWARD LEADERBOARD (10-iter trial, 8 eps/eval)')
print('='*60)
print(f'{"iter":>5} | {"held-out acc":>12} | {"delta vs baseline":>18}')
for row in results:
    delta = (row['held_out_acc'] - baseline_acc) * 100
    sign = '+' if delta >= 0 else ''
    print(f'{row["iteration"]:>5} | {row["held_out_acc"]:>12.1%} | {sign}{delta:>17.1f} pp')

## 6. Read the result — what to look for

**Signal (worth running a longer trial):**
- iter 5 or iter 10 held-out ≥ 30% (vs 20% baseline)
- AND training log shows `real_code=X/4` averaging ≥ 2/4 (was 0.3/4 in Exp B)

→ The shaped reward is breaking the fallback attractor. Worth a 50-iter run later.

**Partial signal:**
- `real_code` goes up but held-out stays flat at 20-25%

→ Attractor broken, but generalization hasn't recovered. Still more honest for slides than the current deck.

**No signal:**
- Everything at 20% ±1 ep

→ Fall back to current slides. No harm done, ~20 min lost.

**Important:** at n=8 eps, 1 flipped episode = 12.5 pp. Treat anything within 12 pp as noise. Only count 25+ pp moves as real signal.

## 7. Download the leaderboard + log

In [ ]:
from google.colab import files
try:
    files.download('shaped_eval_leaderboard.json')
    files.download('shaped_run.log')
except Exception as e:
    print(e)